# Lab01S02 — Consulta GraphQL aos 1.000 repositórios mais populares do GitHub

| RQ | Pergunta | Métrica |
|:--:|----------|--------|
| 01 | Sistemas populares são maduros/antigos? | Idade do repositório |
| 02 | Recebem muita contribuição externa? | Total de PRs aceitas |
| 03 | Lançam releases com frequência? | Total de releases |
| 04 | São atualizados com frequência? | Tempo até última atualização |
| 05 | São escritos nas linguagens mais populares? | Linguagem primária |
| 06 | Possuem alto percentual de issues fechadas? | Razão issues fechadas / total |
| 07 | Sistemas em linguagens populares recebem mais contribuição, releases e updates? | PRs, releases e updates por grupo de linguagem |

## 1. Setup

In [ ]:
import requests
import json
import time
from datetime import datetime, timezone
import os

import pandas as pd

GITHUB_TOKEN = ""  # <-- Insira seu token aqui
GITHUB_API = "https://api.github.com/graphql"
HEADERS = {"Authorization": f"Bearer {GITHUB_TOKEN}"}

assert GITHUB_TOKEN, "Defina seu GITHUB_TOKEN antes de continuar!"
print("Token OK")

Token OK


## 2. Query GraphQL

In [2]:
QUERY = """
{
  search(query: "stars:>1", type: REPOSITORY, first: 10, after: CURSOR) {
    pageInfo { hasNextPage endCursor }
    nodes {
      ... on Repository {
        nameWithOwner
        stargazerCount
        createdAt
        updatedAt
        primaryLanguage { name }
        mergedPRs: pullRequests(states: MERGED) { totalCount }
        releases { totalCount }
        allIssues: issues { totalCount }
        closedIssues: issues(states: CLOSED) { totalCount }
      }
    }
  }
}
"""

## 3. Requisição automática (com retry)

In [3]:
MAX_RETRIES = 5

def fetch_graphql(query_str):
    for attempt in range(1, MAX_RETRIES + 1):
        r = requests.post(GITHUB_API, json={"query": query_str}, headers=HEADERS)
        if r.status_code == 200:
            data = r.json()
            if "errors" in data:
                raise RuntimeError(f"GraphQL errors: {data['errors']}")
            return data
        if r.status_code in (502, 503, 504) and attempt < MAX_RETRIES:
            wait = 2 ** attempt
            print(f"  [retry {attempt}] HTTP {r.status_code} — aguardando {wait}s...")
            time.sleep(wait)
        else:
            r.raise_for_status()

## 4. Coleta dos 1.000 repositórios (paginação)

In [4]:
from tqdm.auto import tqdm

TARGET = 1000
nodes_raw = []
cursor = None

pbar = tqdm(total=TARGET, desc="Coletando repos", unit="repo")

while len(nodes_raw) < TARGET:
    q = QUERY.replace("after: CURSOR", f'after: \"{cursor}\"' if cursor else "")
    result = fetch_graphql(q)
    page = result["data"]["search"]
    new_nodes = page["nodes"]
    nodes_raw.extend(new_nodes)
    pbar.update(len(new_nodes))

    if not page["pageInfo"]["hasNextPage"]:
        break
    cursor = page["pageInfo"]["endCursor"]
    time.sleep(3)

pbar.close()

nodes_raw = nodes_raw[:TARGET]
print(f"\nColeta finalizada: {len(nodes_raw)} repositórios")


Coletando repos:   0%|          | 0/1000 [00:00<?, ?repo/s]


Coleta finalizada: 1000 repositórios


## 5. Montar DataFrame com pandas

In [7]:
now = datetime.now(timezone.utc)

rows = []
for r in nodes_raw:
    created = pd.to_datetime(r["createdAt"])
    updated = pd.to_datetime(r["updatedAt"])
    total = r["allIssues"]["totalCount"]
    closed = r["closedIssues"]["totalCount"]

    rows.append({
        "nome":              r["nameWithOwner"],
        "estrelas":          r["stargazerCount"],
        "criado_em":         created,
        "idade_dias":        (now - created).days,
        "atualizado_em":     updated,
        "dias_desde_update": (now - updated).days,
        "linguagem":         r["primaryLanguage"]["name"] if r["primaryLanguage"] else None,
        "prs_aceitas":       r["mergedPRs"]["totalCount"],
        "releases":          r["releases"]["totalCount"],
        "total_issues":      total,
        "issues_fechadas":   closed,
        "razao_fechadas":    round(closed / total, 4) if total > 0 else 0.0,
    })

df = pd.DataFrame(rows)
df.index += 1
df.index.name = "#"

print(f"DataFrame criado: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head(10)

DataFrame criado: 1000 linhas x 12 colunas


,nome,estrelas,criado_em,idade_dias,atualizado_em,dias_desde_update,linguagem,prs_aceitas,releases,total_issues,issues_fechadas,razao_fechadas
#,,,,,,,,,,,,
1,codecrafters-io/build-your-own-x,472883,2018-05-09 12:03:18+00:00,2858,2026-03-06 20:23:59+00:00,0,Markdown,155,0,888,653,0.7354
2,sindresorhus/awesome,443097,2014-07-11 13:42:37+00:00,4256,2026-03-06 20:35:49+00:00,0,None,694,0,365,349,0.9562
3,freeCodeCamp/freeCodeCamp,437844,2014-12-24 17:49:19+00:00,4090,2026-03-06 20:14:08+00:00,0,TypeScript,27781,0,21157,20983,0.9918
4,public-apis/public-apis,404753,2016-03-20 23:49:42+00:00,3637,2026-03-06 20:33:07+00:00,0,Python,1872,0,839,833,0.9928
5,EbookFoundation/free-programming-books,383678,2013-10-11 06:50:37+00:00,4529,2026-03-06 20:17:20+00:00,0,Python,7333,0,1264,1228,0.9715
6,kamranahmedse/developer-roadmap,350324,2017-03-15 13:45:52+00:00,3278,2026-03-06 20:34:11+00:00,0,TypeScript,4077,1,3086,3056,0.9903
7,jwasham/coding-interview-university,337639,2016-06-06 02:34:12+00:00,3560,2026-03-06 19:46:09+00:00,0,None,415,0,525,454,0.8648
8,donnemartin/system-design-primer,337611,2017-02-26 16:15:28+00:00,3295,2026-03-06 20:29:08+00:00,0,Python,203,0,360,110,0.3056
9,vinta/awesome-python,285895,2014-06-27 21:00:06+00:00,4269,2026-03-06 20:40:36+00:00,0,Python,641,0,0,0,0.0000


## 6. Salvar os dados em arquivos JSON e CSV

In [9]:
os.makedirs("data", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"data/repos_lab01_s02{timestamp}"

df.to_json(f"{filename}.json", orient="records", indent=2, force_ascii=False, date_format="iso")
print(f"Salvo: {filename}.json")

df.to_csv(f"{filename}.csv", index=False)
print(f"Salvo: {filename}.csv")

Salvo: data/repos_lab01_s0220260306_175638.json
Salvo: data/repos_lab01_s0220260306_175638.csv
